In [ ]:
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from joblib import Parallel, delayed
import Conor as c


In [ ]:
def compute_regret_slopes(results, scalings, v_star, fit_start=500):
    """Fit log-log regret slope for each scaling. Returns DataFrame + regret dict."""
    records, regret_dict = [], {}
    for s in scalings:
        label   = f'scaling={s}'
        vpi     = results[label]['vpi']
        eps     = np.arange(1, len(vpi) + 1)
        cum_reg = np.cumsum(np.maximum(v_star - vpi, 0.0))
        regret_dict[label] = cum_reg
        mask = (eps >= fit_start) & (cum_reg > 0)
        if mask.sum() < 10:
            records.append({'scaling': s, 'alpha': float('nan'), 'r_squared': float('nan')})
            continue
        slope, _, r, _, _ = stats.linregress(np.log(eps[mask]), np.log(cum_reg[mask]))
        records.append({'scaling': s, 'alpha': round(slope, 4), 'r_squared': round(r**2, 4)})
    return pd.DataFrame(records), regret_dict


# Reward 1 — asymmetric quadratic

In [ ]:
importlib.reload(c)

cfg_r1 = c.ExpConfig(
    starting_state=4.0,
    action_lo=-5.0,
    action_hi=5.0,
    rho=10.0,
    reward_step_fn=c.reward_quadratic_asymmetric,
)
solver_r1 = c.BellmanSolverScalar(cfg_r1, n_state=201, n_action=201, n_quad=41)
solver_r1.solve()
V_STAR_R1 = solver_r1.get_value(cfg_r1.starting_state)
print(f'V*(x0) = {V_STAR_R1:.4f}  (expected ≈ 0.0)')


In [ ]:
scalings_r1 = [0.01, 0.05, 0.1, 0.5, 1, 5, 10, 50, 100, 500]

configs = [
    c.ExpConfig(
        starting_state=4.0, action_lo=-5.0, action_hi=5.0,
        initial_q=100.0, rho=10.0,
        reward_step_fn=c.reward_quadratic_asymmetric,
        label=f'scaling={s}', scaling=s, alpha=0.5, nEps=2000, n_seeds=10,
    )
    for s in scalings_r1
]
results_sweep_r1 = c.run_experiment(configs, n_jobs=-1)


In [ ]:
summary_r1 = []
for s in scalings_r1:
    label = f'scaling={s}'
    vpi  = results_sweep_r1[label]['vpi']
    arms = results_sweep_r1[label]['arms']
    summary_r1.append({
        'scaling': s,
        'final_cum_reward':  np.sum(vpi),
        'late_mean_reward':  np.mean(vpi[-100:]),
        'final_arms':        arms[-1],
        'mean_arms_last100': np.mean(arms[-100:]),
    })
summary_df_r1 = pd.DataFrame(summary_r1)
summary_df_r1


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(summary_df_r1['scaling'], summary_df_r1['late_mean_reward'], marker='o', color='steelblue', lw=1.5)
ax.axhline(V_STAR_R1, color='black', ls='--', lw=1.2, label=f'V* = {V_STAR_R1:.2f}')
ax.set_xscale('log')
ax.set_xlabel('Scaling (log scale)', fontsize=12)
ax.set_ylabel('Mean reward — last 100 episodes', fontsize=12)
ax.set_title('Performance vs scaling — Reward 1 — asymmetric quadratic', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(summary_df_r1['scaling'], summary_df_r1['final_arms'], marker='o', color='darkorange', lw=1.5)
ax.set_xscale('log')
ax.set_xlabel('Scaling (log scale)', fontsize=12)
ax.set_ylabel('Final number of active balls', fontsize=12)
ax.set_title('Partition growth vs scaling — Reward 1 — asymmetric quadratic', fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(summary_df_r1['final_arms'], summary_df_r1['late_mean_reward'], color='steelblue', s=60)
for _, row in summary_df_r1.iterrows():
    ax.annotate(str(row['scaling']), (row['final_arms'], row['late_mean_reward']),
                textcoords='offset points', xytext=(5, 3), fontsize=9)
ax.axhline(V_STAR_R1, color='black', ls='--', lw=1.2, label=f'V* = {V_STAR_R1:.2f}')
ax.set_xlabel('Final number of active balls', fontsize=12)
ax.set_ylabel('Mean reward — last 100 episodes', fontsize=12)
ax.set_title('Performance vs complexity — Reward 1 — asymmetric quadratic', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
chosen_scalings_r1 = [0.1, 5, 100]

configs = [
    c.ExpConfig(
        starting_state=4.0, action_lo=-5.0, action_hi=5.0,
        initial_q=100.0, rho=10.0,
        reward_step_fn=c.reward_quadratic_asymmetric,
        label=f'scaling={s}', scaling=s, alpha=0.5, nEps=2000, n_seeds=10,
    )
    for s in chosen_scalings_r1
]
results_chosen_r1 = c.run_experiment(configs, n_jobs=-1)


In [ ]:
COLOURS = {0.1: 'steelblue', 5: 'crimson', 100: 'darkorange'}

fig, ax = plt.subplots(figsize=(10, 6))
for s in chosen_scalings_r1:
    ax.plot(results_chosen_r1[f'scaling={s}']['vpi'], label=f'scaling={s}', color=COLOURS[s], lw=1.3)
ax.axhline(V_STAR_R1, color='black', ls='--', lw=1.4, label=f'V* = {V_STAR_R1:.2f}')
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Mean episode reward', fontsize=12)
ax.set_title('Learning curves — Reward 1 — asymmetric quadratic', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for s in chosen_scalings_r1:
    ax.plot(results_chosen_r1[f'scaling={s}']['arms'], label=f'scaling={s}', color=COLOURS[s], lw=1.3)
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Number of active balls', fontsize=12)
ax.set_title('Partition growth over episodes — Reward 1 — asymmetric quadratic', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for s in chosen_scalings_r1:
    vpi     = results_chosen_r1[f'scaling={s}']['vpi']
    eps     = np.arange(1, len(vpi) + 1)
    cum_reg = np.cumsum(np.maximum(V_STAR_R1 - vpi, 0.0))
    mask    = (eps >= 500) & (cum_reg > 0)
    lx, ly  = np.log(eps[mask]), np.log(cum_reg[mask])
    slope, intercept, _, _, _ = stats.linregress(lx, ly)
    ax.plot(lx, ly, lw=1.5, color=COLOURS[s], label=f'scaling={s}  α={slope:.3f}')
    ax.plot(lx, slope*lx + intercept, lw=1.0, color=COLOURS[s], ls=':', alpha=0.8)

# Theoretical worst-case bound α=0.75 anchored through midpoint of best curve
_best_vpi = results_chosen_r1['scaling=5']['vpi']
_eps      = np.arange(1, len(_best_vpi) + 1)
_cum      = np.cumsum(np.maximum(V_STAR_R1 - _best_vpi, 0.0))
_mask     = (_eps >= 500) & (_cum > 0)
_mid      = len(_eps[_mask]) // 2
_th_ic    = np.log(max(_cum[_mask][_mid], 1e-8)) - 0.75 * np.log(_eps[_mask][_mid])
_lx_th    = np.array([np.log(500), np.log(len(_best_vpi))])
ax.plot(_lx_th, 0.75*_lx_th + _th_ic, lw=1.2, color='gray', ls='-.', alpha=0.9,
        label='Theoretical bound  α=0.75')

ax.set_xlim(left=np.log(500))
ax.set_xlabel('log(episode)', fontsize=12)
ax.set_ylabel('log(cumulative regret)', fontsize=12)
ax.set_title('Log-log regret — Reward 1 — asymmetric quadratic', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
slopes_r1, _ = compute_regret_slopes(results_sweep_r1, scalings_r1, V_STAR_R1)
print(slopes_r1.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(slopes_r1['scaling'], slopes_r1['alpha'], marker='o', color='steelblue', lw=1.5, ms=7)
ax.axhline(0.75, color='gray',  ls='-.', lw=1.2, label='Theoretical bound α=0.75  (Theorem 5.19)')
ax.axhline(1.0,  color='black', ls='--', lw=0.9, alpha=0.5, label='Linear regret α=1')
ax.set_xscale('log')
ax.set_ylim(0, 1.15)
ax.set_xlabel('Scaling (log scale)', fontsize=12)
ax.set_ylabel('Regret exponent α', fontsize=12)
ax.set_title('Regret exponent vs scaling — Reward 1 — asymmetric quadratic', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# Reward 2 — shifted quadratic

In [ ]:
importlib.reload(c)

cfg_r2 = c.ExpConfig(
    starting_state=4.0,
    action_lo=-5.0,
    action_hi=5.0,
    rho=10.0,
    reward_step_fn=c.reward_quadratic_shifted,
)
solver_r2 = c.BellmanSolverScalar(cfg_r2, n_state=201, n_action=201, n_quad=41)
solver_r2.solve()
V_STAR_R2 = solver_r2.get_value(cfg_r2.starting_state)
print(f'V*(x0) = {V_STAR_R2:.4f}  (expected ≈ 100.0)')


In [ ]:
scalings_r2 = [0.01, 0.05, 0.1, 0.5, 1, 5, 10, 50, 100, 500]

configs = [
    c.ExpConfig(
        starting_state=4.0, action_lo=-5.0, action_hi=5.0,
        initial_q=100.0, rho=10.0,
        reward_step_fn=c.reward_quadratic_shifted,
        label=f'scaling={s}', scaling=s, alpha=0.5, nEps=2000, n_seeds=10,
    )
    for s in scalings_r2
]
results_sweep_r2 = c.run_experiment(configs, n_jobs=-1)


In [ ]:
summary_r2 = []
for s in scalings_r2:
    label = f'scaling={s}'
    vpi  = results_sweep_r2[label]['vpi']
    arms = results_sweep_r2[label]['arms']
    summary_r2.append({
        'scaling': s,
        'final_cum_reward':  np.sum(vpi),
        'late_mean_reward':  np.mean(vpi[-100:]),
        'final_arms':        arms[-1],
        'mean_arms_last100': np.mean(arms[-100:]),
    })
summary_df_r2 = pd.DataFrame(summary_r2)
summary_df_r2


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(summary_df_r2['scaling'], summary_df_r2['late_mean_reward'], marker='o', color='steelblue', lw=1.5)
ax.axhline(V_STAR_R2, color='black', ls='--', lw=1.2, label=f'V* = {V_STAR_R2:.2f}')
ax.set_xscale('log')
ax.set_xlabel('Scaling (log scale)', fontsize=12)
ax.set_ylabel('Mean reward — last 100 episodes', fontsize=12)
ax.set_title('Performance vs scaling — Reward 2 — shifted quadratic', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(summary_df_r2['scaling'], summary_df_r2['final_arms'], marker='o', color='darkorange', lw=1.5)
ax.set_xscale('log')
ax.set_xlabel('Scaling (log scale)', fontsize=12)
ax.set_ylabel('Final number of active balls', fontsize=12)
ax.set_title('Partition growth vs scaling — Reward 2 — shifted quadratic', fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(summary_df_r2['final_arms'], summary_df_r2['late_mean_reward'], color='steelblue', s=60)
for _, row in summary_df_r2.iterrows():
    ax.annotate(str(row['scaling']), (row['final_arms'], row['late_mean_reward']),
                textcoords='offset points', xytext=(5, 3), fontsize=9)
ax.axhline(V_STAR_R2, color='black', ls='--', lw=1.2, label=f'V* = {V_STAR_R2:.2f}')
ax.set_xlabel('Final number of active balls', fontsize=12)
ax.set_ylabel('Mean reward — last 100 episodes', fontsize=12)
ax.set_title('Performance vs complexity — Reward 2 — shifted quadratic', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
chosen_scalings_r2 = [0.1, 5, 100]

configs = [
    c.ExpConfig(
        starting_state=4.0, action_lo=-5.0, action_hi=5.0,
        initial_q=100.0, rho=10.0,
        reward_step_fn=c.reward_quadratic_shifted,
        label=f'scaling={s}', scaling=s, alpha=0.5, nEps=2000, n_seeds=10,
    )
    for s in chosen_scalings_r2
]
results_chosen_r2 = c.run_experiment(configs, n_jobs=-1)


In [ ]:
COLOURS = {0.1: 'steelblue', 5: 'crimson', 100: 'darkorange'}

fig, ax = plt.subplots(figsize=(10, 6))
for s in chosen_scalings_r2:
    ax.plot(results_chosen_r2[f'scaling={s}']['vpi'], label=f'scaling={s}', color=COLOURS[s], lw=1.3)
ax.axhline(V_STAR_R2, color='black', ls='--', lw=1.4, label=f'V* = {V_STAR_R2:.2f}')
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Mean episode reward', fontsize=12)
ax.set_title('Learning curves — Reward 2 — shifted quadratic', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for s in chosen_scalings_r2:
    ax.plot(results_chosen_r2[f'scaling={s}']['arms'], label=f'scaling={s}', color=COLOURS[s], lw=1.3)
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Number of active balls', fontsize=12)
ax.set_title('Partition growth over episodes — Reward 2 — shifted quadratic', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for s in chosen_scalings_r2:
    vpi     = results_chosen_r2[f'scaling={s}']['vpi']
    eps     = np.arange(1, len(vpi) + 1)
    cum_reg = np.cumsum(np.maximum(V_STAR_R2 - vpi, 0.0))
    mask    = (eps >= 500) & (cum_reg > 0)
    lx, ly  = np.log(eps[mask]), np.log(cum_reg[mask])
    slope, intercept, _, _, _ = stats.linregress(lx, ly)
    ax.plot(lx, ly, lw=1.5, color=COLOURS[s], label=f'scaling={s}  α={slope:.3f}')
    ax.plot(lx, slope*lx + intercept, lw=1.0, color=COLOURS[s], ls=':', alpha=0.8)

# Theoretical worst-case bound α=0.75 anchored through midpoint of best curve
_best_vpi = results_chosen_r2['scaling=5']['vpi']
_eps      = np.arange(1, len(_best_vpi) + 1)
_cum      = np.cumsum(np.maximum(V_STAR_R2 - _best_vpi, 0.0))
_mask     = (_eps >= 500) & (_cum > 0)
_mid      = len(_eps[_mask]) // 2
_th_ic    = np.log(max(_cum[_mask][_mid], 1e-8)) - 0.75 * np.log(_eps[_mask][_mid])
_lx_th    = np.array([np.log(500), np.log(len(_best_vpi))])
ax.plot(_lx_th, 0.75*_lx_th + _th_ic, lw=1.2, color='gray', ls='-.', alpha=0.9,
        label='Theoretical bound  α=0.75')

ax.set_xlim(left=np.log(500))
ax.set_xlabel('log(episode)', fontsize=12)
ax.set_ylabel('log(cumulative regret)', fontsize=12)
ax.set_title('Log-log regret — Reward 2 — shifted quadratic', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
slopes_r2, _ = compute_regret_slopes(results_sweep_r2, scalings_r2, V_STAR_R2)
print(slopes_r2.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(slopes_r2['scaling'], slopes_r2['alpha'], marker='o', color='steelblue', lw=1.5, ms=7)
ax.axhline(0.75, color='gray',  ls='-.', lw=1.2, label='Theoretical bound α=0.75  (Theorem 5.19)')
ax.axhline(1.0,  color='black', ls='--', lw=0.9, alpha=0.5, label='Linear regret α=1')
ax.set_xscale('log')
ax.set_ylim(0, 1.15)
ax.set_xlabel('Scaling (log scale)', fontsize=12)
ax.set_ylabel('Regret exponent α', fontsize=12)
ax.set_title('Regret exponent vs scaling — Reward 2 — shifted quadratic', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# Reward 3 — quartic

In [ ]:
importlib.reload(c)

cfg_r3 = c.ExpConfig(
    starting_state=4.0,
    action_lo=-5.0,
    action_hi=5.0,
    rho=10.0,
    reward_step_fn=c.reward_quartic,
)
solver_r3 = c.BellmanSolverScalar(cfg_r3, n_state=201, n_action=201, n_quad=41)
solver_r3.solve()
V_STAR_R3 = solver_r3.get_value(cfg_r3.starting_state)
print(f'V*(x0) = {V_STAR_R3:.4f}  (expected ≈ 0.0)')


In [ ]:
scalings_r3 = [0.01, 0.05, 0.1, 0.5, 1, 5, 10, 50, 100, 500]

configs = [
    c.ExpConfig(
        starting_state=4.0, action_lo=-5.0, action_hi=5.0,
        initial_q=100.0, rho=10.0,
        reward_step_fn=c.reward_quartic,
        label=f'scaling={s}', scaling=s, alpha=0.5, nEps=2000, n_seeds=10,
    )
    for s in scalings_r3
]
results_sweep_r3 = c.run_experiment(configs, n_jobs=-1)


In [ ]:
summary_r3 = []
for s in scalings_r3:
    label = f'scaling={s}'
    vpi  = results_sweep_r3[label]['vpi']
    arms = results_sweep_r3[label]['arms']
    summary_r3.append({
        'scaling': s,
        'final_cum_reward':  np.sum(vpi),
        'late_mean_reward':  np.mean(vpi[-100:]),
        'final_arms':        arms[-1],
        'mean_arms_last100': np.mean(arms[-100:]),
    })
summary_df_r3 = pd.DataFrame(summary_r3)
summary_df_r3


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(summary_df_r3['scaling'], summary_df_r3['late_mean_reward'], marker='o', color='steelblue', lw=1.5)
ax.axhline(V_STAR_R3, color='black', ls='--', lw=1.2, label=f'V* = {V_STAR_R3:.2f}')
ax.set_xscale('log')
ax.set_xlabel('Scaling (log scale)', fontsize=12)
ax.set_ylabel('Mean reward — last 100 episodes', fontsize=12)
ax.set_title('Performance vs scaling — Reward 3 — quartic', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(summary_df_r3['scaling'], summary_df_r3['final_arms'], marker='o', color='darkorange', lw=1.5)
ax.set_xscale('log')
ax.set_xlabel('Scaling (log scale)', fontsize=12)
ax.set_ylabel('Final number of active balls', fontsize=12)
ax.set_title('Partition growth vs scaling — Reward 3 — quartic', fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(summary_df_r3['final_arms'], summary_df_r3['late_mean_reward'], color='steelblue', s=60)
for _, row in summary_df_r3.iterrows():
    ax.annotate(str(row['scaling']), (row['final_arms'], row['late_mean_reward']),
                textcoords='offset points', xytext=(5, 3), fontsize=9)
ax.axhline(V_STAR_R3, color='black', ls='--', lw=1.2, label=f'V* = {V_STAR_R3:.2f}')
ax.set_xlabel('Final number of active balls', fontsize=12)
ax.set_ylabel('Mean reward — last 100 episodes', fontsize=12)
ax.set_title('Performance vs complexity — Reward 3 — quartic', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
chosen_scalings_r3 = [0.1, 5, 100]

configs = [
    c.ExpConfig(
        starting_state=4.0, action_lo=-5.0, action_hi=5.0,
        initial_q=100.0, rho=10.0,
        reward_step_fn=c.reward_quartic,
        label=f'scaling={s}', scaling=s, alpha=0.5, nEps=2000, n_seeds=10,
    )
    for s in chosen_scalings_r3
]
results_chosen_r3 = c.run_experiment(configs, n_jobs=-1)


In [ ]:
COLOURS = {0.1: 'steelblue', 5: 'crimson', 100: 'darkorange'}

fig, ax = plt.subplots(figsize=(10, 6))
for s in chosen_scalings_r3:
    ax.plot(results_chosen_r3[f'scaling={s}']['vpi'], label=f'scaling={s}', color=COLOURS[s], lw=1.3)
ax.axhline(V_STAR_R3, color='black', ls='--', lw=1.4, label=f'V* = {V_STAR_R3:.2f}')
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Mean episode reward', fontsize=12)
ax.set_title('Learning curves — Reward 3 — quartic', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for s in chosen_scalings_r3:
    ax.plot(results_chosen_r3[f'scaling={s}']['arms'], label=f'scaling={s}', color=COLOURS[s], lw=1.3)
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Number of active balls', fontsize=12)
ax.set_title('Partition growth over episodes — Reward 3 — quartic', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for s in chosen_scalings_r3:
    vpi     = results_chosen_r3[f'scaling={s}']['vpi']
    eps     = np.arange(1, len(vpi) + 1)
    cum_reg = np.cumsum(np.maximum(V_STAR_R3 - vpi, 0.0))
    mask    = (eps >= 500) & (cum_reg > 0)
    lx, ly  = np.log(eps[mask]), np.log(cum_reg[mask])
    slope, intercept, _, _, _ = stats.linregress(lx, ly)
    ax.plot(lx, ly, lw=1.5, color=COLOURS[s], label=f'scaling={s}  α={slope:.3f}')
    ax.plot(lx, slope*lx + intercept, lw=1.0, color=COLOURS[s], ls=':', alpha=0.8)

# Theoretical worst-case bound α=0.75 anchored through midpoint of best curve
_best_vpi = results_chosen_r3['scaling=5']['vpi']
_eps      = np.arange(1, len(_best_vpi) + 1)
_cum      = np.cumsum(np.maximum(V_STAR_R3 - _best_vpi, 0.0))
_mask     = (_eps >= 500) & (_cum > 0)
_mid      = len(_eps[_mask]) // 2
_th_ic    = np.log(max(_cum[_mask][_mid], 1e-8)) - 0.75 * np.log(_eps[_mask][_mid])
_lx_th    = np.array([np.log(500), np.log(len(_best_vpi))])
ax.plot(_lx_th, 0.75*_lx_th + _th_ic, lw=1.2, color='gray', ls='-.', alpha=0.9,
        label='Theoretical bound  α=0.75')

ax.set_xlim(left=np.log(500))
ax.set_xlabel('log(episode)', fontsize=12)
ax.set_ylabel('log(cumulative regret)', fontsize=12)
ax.set_title('Log-log regret — Reward 3 — quartic', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
slopes_r3, _ = compute_regret_slopes(results_sweep_r3, scalings_r3, V_STAR_R3)
print(slopes_r3.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(slopes_r3['scaling'], slopes_r3['alpha'], marker='o', color='steelblue', lw=1.5, ms=7)
ax.axhline(0.75, color='gray',  ls='-.', lw=1.2, label='Theoretical bound α=0.75  (Theorem 5.19)')
ax.axhline(1.0,  color='black', ls='--', lw=0.9, alpha=0.5, label='Linear regret α=1')
ax.set_xscale('log')
ax.set_ylim(0, 1.15)
ax.set_xlabel('Scaling (log scale)', fontsize=12)
ax.set_ylabel('Regret exponent α', fontsize=12)
ax.set_title('Regret exponent vs scaling — Reward 3 — quartic', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# Cross-reward comparison

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

for slopes, label, colour in [
    (slopes_r1, 'Asymmetric quadratic (m=1)', 'steelblue'),
    (slopes_r2, 'Shifted quadratic (m=1)',    'crimson'),
    (slopes_r3, 'Quartic (m=3)',              'darkorange'),
]:
    ax.plot(slopes['scaling'], slopes['alpha'], marker='o', lw=1.5, label=label)

ax.axhline(0.75, color='gray',  ls='-.', lw=1.2, label='Theoretical bound α=0.75')
ax.axhline(1.0,  color='black', ls='--', lw=0.9, alpha=0.5, label='Linear regret α=1')
ax.set_xscale('log')
ax.set_ylim(0, 1.15)
ax.set_xlabel('Scaling (log scale)', fontsize=12)
ax.set_ylabel('Regret exponent α', fontsize=12)
ax.set_title('Regret exponent vs scaling — cross-reward comparison\n'
             'Quartic (m=3) predicted to show higher α → larger zooming dimension', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
